In [1]:
from pathlib import Path
import sys,json,hashlib,importlib.util
ROOT=Path("/home/featurize/work/biological_prior_validation").resolve()
FIT_ZIP=ROOT/"results/summary/task2_step3_review_bundle_task2_ordered_controls_num_v3.zip"
REPORT_ZIP=ROOT/"results/summary/task2_three_comparisons_v1_review_bundle.zip"
CONTEXT_ZIP=ROOT/"results/summary/task2_context_reproducibility_v1_review_bundle.zip"
ANALYSIS_NAME="task2_four_figures_table1_v1"
if not ROOT.is_dir():raise FileNotFoundError(f"Please modify ROOT:{ROOT}")
for p in [FIT_ZIP,REPORT_ZIP,CONTEXT_ZIP]:
    if not p.is_file():raise FileNotFoundError(f"Missing freeze input:{p}")


In [2]:
SOURCES = {'frozen_task2_reporting': '"""Task 2.4: immutable score reporting; no count/target reads or model fitting."""\nfrom pathlib import Path\nfrom collections import Counter\nimport json, hashlib, zipfile, io, sys, subprocess\nimport numpy as np\nimport pandas as pd\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nMETRICS=[\'mean_poisson_deviance\',\'normalized_mse\']\nMETA=[\'scope_id\',\'cell_line\',\'drug\',\'replicate\',\'n_cells\',\'n_pairs\']\n\ndef sha(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for block in iter(lambda:f.read(1024*1024),b\'\'):h.update(block)\n return h.hexdigest()\ndef json_bytes(obj):return (json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\').encode()\ndef write_same(path,data):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n if path.exists() and path.read_bytes()!=data:raise ValueError(\'Different existing output; use a new report version: \'+str(path))\n if not path.exists():path.write_bytes(data)\ndef relative(delta,reference):return float(delta/reference*100) if reference>0 else np.nan\ndef validate_frame(df):\n if len(df)!=1290 or set(df.scope_id)!=set(range(30)) or not df.status.eq(\'OK\').all():raise ValueError(\'Complete all15 OK score table required\')\n if not np.isfinite(df[METRICS].to_numpy(float)).all() or (df[METRICS]<0).any().any():raise ValueError(\'Invalid loss values\')\n for key in [\'scope_id\',\'n_cells\',\'n_pairs\']:\n  v=df[key].to_numpy(float)\n  if not np.isfinite(v).all() or (v!=np.floor(v)).any() or (v<0).any():raise ValueError(\'Invalid integer identity/count: \'+key)\n seed=df.loc[df.method.isin([\'M3\',\'M4\']),\'seed_index\'].to_numpy(float)\n if not np.isfinite(seed).all() or (seed!=np.floor(seed)).any():raise ValueError(\'Invalid seed indices\')\n if df.loc[df.method.isin([\'M0\',\'M1\',\'M2\']),\'seed_index\'].notna().any():raise ValueError(\'Unexpected seed for deterministic method\')\n for sid,g in df.groupby(\'scope_id\'):\n  if g.method.value_counts().to_dict()!={\'M3\':20,\'M4\':20,\'M0\':1,\'M1\':1,\'M2\':1}:raise ValueError(\'Method/seed roster mismatch\')\n  for m in [\'M3\',\'M4\']:\n   if sorted(g.loc[g.method.eq(m),\'seed_index\'].astype(int))!=list(range(20)):raise ValueError(\'Incomplete seed indices\')\n  for c in META[1:]+[\'gene_sha256\',\'Lc_sha256\',\'common_I_row_sha256\']:\n   if g[c].nunique(dropna=False)!=1 or g[c].isna().any():raise ValueError(\'Common-I metadata mismatch: \'+c)\n  r=g.loc[g.method.eq(\'M2\')].iloc[0];m3=g.loc[g.method.eq(\'M3\')]\n  for c in [\'alpha\',\'beta\',\'trace_Lg\',\'weight_sum\',\'zero_weight_edges\']:\n   if not np.allclose(m3[c].to_numpy(float),float(r[c]),rtol=1e-10,atol=1e-12):raise ValueError(\'Matched control not matched: \'+c)\n  if not m3.lambda_rule.eq(r.lambda_rule).all():raise ValueError(\'Matched parameters mismatch\')\n\ndef read_verified_bundle(path):\n with zipfile.ZipFile(path) as z:\n  if len(set(z.namelist()))!=len(z.namelist()) or z.testzip() is not None:raise ValueError(\'Corrupt/duplicate ZIP entries\')\n  js=lambda n:json.loads(z.read(n))\n  manifest=js(\'review_manifest.json\')\n  for item in manifest[\'files\']:\n   if hashlib.sha256(z.read(item[\'name\'])).hexdigest()!=item[\'sha256\']:raise ValueError(\'Manifest mismatch: \'+item[\'name\'])\n  contract=js(\'experiment_contract.json\');effective=js(\'effective_parameter_grid.json\');summary=js(\'task2_step3_summary.json\')\n  if not summary[\'full15_run\'] or js(\'all15_summary.json\')[\'status\']!=\'all15_exploratory_completed\':raise ValueError(\'All15 run incomplete\')\n  if summary[\'old_lock_before\']!=summary[\'old_lock_after\']:raise ValueError(\'Old target lock changed\')\n  if hashlib.sha256(z.read(\'src/task2_experiments.py\')).hexdigest()!=contract[\'engine_sha256\']:raise ValueError(\'Fitting engine mismatch\')\n  if hashlib.sha256(json.dumps(effective,sort_keys=True).encode()).hexdigest()!=contract[\'effective_grid_sha256\']:raise ValueError(\'Effective grid mismatch\')\n  df=pd.read_csv(io.BytesIO(z.read(\'all15_all_scores.csv\')));validate_frame(df)\n  genes=pd.read_csv(io.BytesIO(z.read(\'outer_gene_ids.csv\')))\n  if not np.allclose(df.n_pairs,df.n_cells*len(genes)):raise ValueError(\'Incomplete pair coverage\')\n  for col,total in [(\'mean_poisson_deviance\',\'deviance_sum\'),(\'normalized_mse\',\'normalized_sse_sum\')]:\n   if not np.allclose(df[col],df[total]/df.n_pairs,rtol=1e-10,atol=1e-12):raise ValueError(\'Score/sufficient-sum mismatch\')\n  for s in contract[\'all_scopes\']:\n   g=df[df.scope_id.eq(s[\'scope_id\'])]\n   if any(not g[c].eq(s[c]).all() for c in META):raise ValueError(\'Contract scope mismatch\')\n   p=f"scopes/scope_{s[\'scope_id\']:02d}/precision/";gate=js(p+\'gate.json\');probes=pd.read_csv(io.BytesIO(z.read(p+\'probes.csv\')))\n   if gate[\'status\']!=\'PASS\' or gate[\'final_B_used\'] is not False or len(probes)!=19 or not probes.status.eq(\'PASS\').all():raise ValueError(\'Real precision gate failed\')\n   for key in [\'relative_prediction_l2\',\'relative_development_deviance\',\'absolute_s_max\']:\n    values=probes.loc[probes.method.eq(\'M2\'),key] if key==\'absolute_s_max\' else probes[key]\n    if not np.isfinite(values).all() or not values.le(effective[\'precision\'][\'thresholds\'][key]).all():raise ValueError(\'Precision values exceed threshold\')\n  for stage in [\'fixed\',\'adaptive\',\'matched\',\'rewired\']:\n   prefix=f\'scores/all15/{stage}/\';ledger=js(prefix+\'target_access_ledger.json\');registry=js(prefix+\'frozen_registry.json\')\n   if ledger[\'registry_sha256\']!=hashlib.sha256(z.read(prefix+\'frozen_registry.json\')).hexdigest() or registry[\'experiment_contract_sha256\']!=hashlib.sha256(z.read(\'experiment_contract.json\')).hexdigest():raise ValueError(\'Registry/ledger linkage mismatch\')\n   expected=[(r[\'scope\'][\'scope_id\'],r[\'method\'],None if r[\'seed_index\'] is None else int(r[\'seed_index\'])) for r in registry[\'records\']]\n   stage_df=pd.read_csv(io.BytesIO(z.read(prefix+\'scores.csv\')))\n   actual=[(r.scope_id,r.method,None if pd.isna(r.seed_index) else int(r.seed_index)) for r in stage_df.itertuples()]\n   if Counter(expected)!=Counter(actual):raise ValueError(\'Stage registry/score coverage mismatch\')\n   compare=df[df.method.isin(stage_df.method.unique())].set_index([\'scope_id\',\'method\',\'seed_index\']).sort_index()\n   stage_df=stage_df.set_index([\'scope_id\',\'method\',\'seed_index\']).sort_index()\n   if not np.allclose(compare[METRICS],stage_df[METRICS],rtol=1e-12,atol=1e-14):raise ValueError(\'Stage scores differ from summary\')\n  failures=[];tuning_counts=Counter();edge_counts={}\n  for n in z.namelist():\n   if \'/inner/tuning/\' in n and n.endswith(\'search_table.csv\'):\n    t=pd.read_csv(io.BytesIO(z.read(n)));method=n.split(\'/inner/tuning/\')[1].split(\'/\')[0].split(\'_\')[0]\n    for status,count in t.status.value_counts().items():tuning_counts[(method,status)]+=int(count)\n    for row in t.loc[~t.status.eq(\'OK\')].to_dict(\'records\'):failures.append({\'source_table\':n,\'method\':method,**row})\n  for sid in df.scope_id.unique():\n   a=js(f\'scopes/scope_{sid:02d}/outer/models/M2/graph_audit.json\');edge_counts[int(sid)]=int(a[\'effective_gene\'][\'support_edges\'])\n  return df,contract,effective,pd.DataFrame(failures),pd.DataFrame([{\'method\':m,\'status\':s,\'n_candidates\':n} for (m,s),n in sorted(tuning_counts.items())]),edge_counts\n\ndef calculate_tables(df,tie_atol=1e-12,tie_rtol=1e-10):\n validate_frame(df)\n if not np.isfinite([tie_atol,tie_rtol]).all() or min(tie_atol,tie_rtol)<0:raise ValueError(\'Invalid tie tolerance\')\n fixed=[];pairs=[];matched=[];null_pairs=[];nulls=[]\n for sid,g in df.groupby(\'scope_id\',sort=True):\n  meta=g[META].iloc[0].to_dict();base={m:g[g.method.eq(m)].iloc[0] for m in [\'M0\',\'M1\',\'M2\']}\n  for metric in METRICS:\n   for label,method,ref in [(\'fixed\',\'M1\',\'M0\'),(\'adaptive\',\'M2\',\'M1\')]:\n    obs=float(base[method][metric]);reference=float(base[ref][metric]);delta=obs-reference\n    fixed.append({**meta,\'metric\':metric,\'comparison\':label,\'method\':method,\'reference_method\':ref,\'score\':obs,\'reference_score\':reference,\'delta_absolute\':delta,\'delta_percent\':relative(delta,reference),\'relative_available\':reference>0})\n   values=[];percents=[]\n   for r in g[g.method.eq(\'M3\')].sort_values(\'seed_index\').to_dict(\'records\'):\n    reference=float(r[metric]);obs=float(base[\'M2\'][metric]);delta=obs-reference;percent=relative(delta,reference);values.append(delta);percents.append(percent)\n    pairs.append({**meta,\'metric\':metric,\'seed_index\':int(r[\'seed_index\']),\'M2_score\':obs,\'M3_score\':reference,\'delta_absolute\':delta,\'delta_percent\':percent,\'relative_available\':reference>0,\'M2_conditional_EDF\':base[\'M2\'][\'conditional_trace_H\'],\'M3_conditional_EDF\':r[\'conditional_trace_H\'],\'conditional_EDF_delta_M2_minus_M3\':base[\'M2\'][\'conditional_trace_H\']-r[\'conditional_trace_H\']})\n   refs=g[g.method.eq(\'M3\')][metric].to_numpy(float);ref_mean=float(refs.mean());obs=float(base[\'M2\'][metric]);delta=obs-ref_mean\n   matched.append({**meta,\'metric\':metric,\'comparison\':\'matched\',\'n_seeds\':len(refs),\'M2_score\':obs,\'M3_mean\':ref_mean,\'M3_median\':float(np.median(refs)),\'M3_q05\':float(np.quantile(refs,.05)),\'M3_q95\':float(np.quantile(refs,.95)),\'delta_absolute_vs_mean\':delta,\'delta_percent_vs_mean\':relative(delta,ref_mean),\'mean_seed_delta_percent\':float(np.nanmean(percents)) if np.isfinite(percents).any() else np.nan,\'delta_seed_q05\':float(np.quantile(values,.05)),\'delta_seed_q95\':float(np.quantile(values,.95)),\'n_M2_better\':int((np.array(values)<0).sum()),\'n_equal_exact\':int((np.array(values)==0).sum()),\'n_M2_worse\':int((np.array(values)>0).sum()),\'relative_available\':ref_mean>0})\n   refs=g[g.method.eq(\'M4\')][metric].to_numpy(float);obs=float(base[\'M1\'][metric]);tol=tie_atol+tie_rtol*abs(obs)\n   lower=int((refs<obs-tol).sum());ties=int((np.abs(refs-obs)<=tol).sum());higher=int((refs>obs+tol).sum());assert lower+ties+higher==len(refs)\n   for r in g[g.method.eq(\'M4\')].sort_values(\'seed_index\').to_dict(\'records\'):\n    ref=float(r[metric]);d=obs-ref;null_pairs.append({**meta,\'metric\':metric,\'seed_index\':int(r[\'seed_index\']),\'M1_score\':obs,\'M4_score\':ref,\'delta_absolute_M1_minus_M4\':d,\'delta_percent_M1_minus_M4\':relative(d,ref),\'relative_available\':ref>0})\n   ref_mean=float(refs.mean());d=obs-ref_mean\n   nulls.append({**meta,\'metric\':metric,\'n_seeds\':len(refs),\'M1_score\':obs,\'M4_mean\':ref_mean,\'M4_median\':float(np.median(refs)),\'M4_q05\':float(np.quantile(refs,.05)),\'M4_q95\':float(np.quantile(refs,.95)),\'delta_absolute_M1_minus_null_mean\':d,\'delta_percent_M1_minus_null_mean\':relative(d,ref_mean),\'n_null_better\':lower,\'n_numerical_ties\':ties,\'n_null_worse\':higher,\'M1_rank_best\':1+lower,\'M1_rank_worst\':1+lower+ties,\'M1_midrank_percentile\':100*(lower+.5*ties)/len(refs),\'tie_score_tolerance\':tol,\'relative_available\':ref_mean>0})\n return {\'fixed_adaptive_effects\':pd.DataFrame(fixed),\'matched_seed_effects\':pd.DataFrame(pairs),\'matched_summary\':pd.DataFrame(matched),\'M4_seed_effects\':pd.DataFrame(null_pairs),\'M4_null_positions\':pd.DataFrame(nulls)}\n\ndef consistency(tables):\n f=tables[\'fixed_adaptive_effects\'].rename(columns={\'delta_absolute\':\'delta\',\'delta_percent\':\'percent\'})\n m=tables[\'matched_summary\'].rename(columns={\'delta_absolute_vs_mean\':\'delta\',\'delta_percent_vs_mean\':\'percent\'})\n records=[]\n for key,g in pd.concat([f,m],ignore_index=True).groupby([\'cell_line\',\'drug\',\'metric\',\'comparison\']):\n  g=g.sort_values(\'replicate\')\n  if len(g)!=2:raise ValueError(\'Exactly two replicate labels per context required\')\n  v=g.delta.to_numpy(float);label=\'both_better\' if (v<0).all() else \'both_worse\' if (v>0).all() else \'both_exact_zero\' if (v==0).all() else \'opposite_or_zero\'\n  records.append(dict(zip([\'cell_line\',\'drug\',\'metric\',\'comparison\'],key),replicate_label_1=g.iloc[0].replicate,replicate_label_2=g.iloc[1].replicate,delta_label_1=v[0],delta_label_2=v[1],percent_label_1=g.iloc[0].percent,percent_label_2=g.iloc[1].percent,sign_consistency=label,interpretation=\'descriptive label consistency, not biological cross-replicate directions\'))\n return pd.DataFrame(records)\n\ndef run_report(input_zip,root,report_name=\'task2_three_comparisons_v1\',tie_atol=1e-12,tie_rtol=1e-10,self_tests=None):\n root=Path(root).resolve();input_zip=Path(input_zip).resolve()\n if Path(report_name).name!=report_name or report_name in [\'\',\'.\',\'..\']:raise ValueError(\'Invalid report name\')\n if self_tests is not None and self_tests.get(\'status\')!=\'PASS\':raise ValueError(\'Self tests must pass\')\n df,source_contract,effective,failed,candidates,edge_counts=read_verified_bundle(input_zip)\n try:git_commit=subprocess.run([\'git\',\'rev-parse\',\'--verify\',\'HEAD\'],cwd=root,capture_output=True,text=True,check=True).stdout.strip()\n except (OSError,subprocess.CalledProcessError):git_commit=None\n config={\'schema\':\'task2-step4-report-v1\',\'input_zip_sha256\':sha(input_zip),\'source_experiment_engine_sha256\':source_contract[\'engine_sha256\'],\'report_engine_sha256\':ENGINE_SHA,\'metrics\':METRICS,\'sign\':\'negative delta means lower loss\',\'formulas\':{\'fixed\':\'D(M1)-D(M0), percentage denominator D(M0)\',\'adaptive\':\'D(M2)-D(M1), denominator D(M1)\',\'matched_seed\':\'D(M2)-D(M3_seed), denominator D(M3_seed)\',\'matched_scope\':\'D(M2)-mean_seed D(M3), denominator mean_seed D(M3)\',\'M4\':\'D(M1)-D(M4_seed), denominator D(M4_seed)\'},\'M3_summary_percent\':\'ratio against mean null loss; distinct from mean of per-seed ratios\',\'tie_atol\':tie_atol,\'tie_rtol\':tie_rtol,\'seed_quantiles\':\'5/95 percentiles of 20 saved randomized runs, descriptive ranges, NOT biological confidence intervals\',\'inference\':\'No formal p-values/multiple-testing claims or cell bootstrap. Independent biological sampling units unverified; biological uncertainty not estimable from this bundle. All contexts retained. Finite graph swaps not proven uniform/mixed.\',\'replicates\':\'Scenario A label-stratified molecular holdout; Scenario B blocked\',\'target_access\':\'Reads saved score/audit ZIP only; no raw B access, fitting, tuning or seed extension\',\'git_commit\':git_commit,\'environment\':{\'python\':sys.version,\'numpy\':np.__version__,\'pandas\':pd.__version__}}\n config_dir=root/\'configs\'/report_name;out=root/\'results/summary\'/report_name;raw=root/\'results/raw\'/report_name;report_dir=root/\'reports\'/report_name\n write_same(config_dir/\'report_contract.json\',json_bytes(config))\n tables=calculate_tables(df,tie_atol,tie_rtol);tables[\'label_sign_consistency\']=consistency(tables)\n tables[\'failed_candidates\']=failed;tables[\'candidate_status_summary\']=candidates\n audit=[]\n for sid,g in df.groupby(\'scope_id\'):\n  r=g[g.method.eq(\'M2\')].iloc[0];m3=g[g.method.eq(\'M3\')];audit.append({**r[META].to_dict(),\'support_edges\':edge_counts[sid],\'M2_zero_edge_fraction\':float(r.zero_weight_edges/edge_counts[sid]) if edge_counts[sid]>0 else np.nan,\'M2_weight_sum\':r.weight_sum,\'M2_trace_Lg\':r.trace_Lg,\'M2_conditional_EDF\':r.conditional_trace_H,\'M3_conditional_EDF_mean\':float(m3.conditional_trace_H.mean()),\'M3_EDF_relative_percent_mean\':float(m3.conditional_trace_H_relative_percent.mean()),\'note\':\'Matched total weight and trace do not imply matched conditional EDF; conditional EDF is not total adaptive DF\'})\n tables[\'smoothing_audit\']=pd.DataFrame(audit)\n write_same(raw/\'all15_complete_benchmark.csv\',df.to_csv(index=False).encode())\n for name,table in tables.items():write_same(out/(name+\'.csv\'),table.to_csv(index=False).encode())\n summary={\'status\':\'COMPLETE\',\'n_contexts\':15,\'n_scope_labels\':30,\'n_score_rows\':len(df),\'n_formal_precision_probes\':570,\'n_failed_candidates\':len(failed),\'new_raw_B_access\':False,\'scenario_B\':\'BLOCKED\',\'source_input_sha256\':config[\'input_zip_sha256\'],\'old_target_lock\':source_contract[\'old_target_history\'][\'target_access_lock_sha256\']}\n write_same(out/\'summary.json\',json_bytes(summary))\n memo=\'任务2.4：三组主要对比\\n\\n负差值表示改善。所有15 contexts和两个replicate labels完整保留。\\nfixed=M1-M0；adaptive=M2-M1；matched=M2-M3。matched逐seed表和null均值摘要同时报告，两种相对百分比聚合不可混淆。M4单独报告M1在随机图分布中的排名和分位数，lower percentile对应更低loss。\\nseed分位数是技术随机化描述，非biological置信区间；不提供正式p值或显著性结论。独立sample/plate/well来源未确认，无法由cell数估计biological不确定性。M3的conditional EDF变化单独保留，不能把matched差异直接归因于biological edge identity。\\n全部失败候选保留。当前报告只读取已冻结的scores/audits，不读取原始B、不重拟合、不重新选参数或seed。重要结果的100-seed扩展属于后续新版本工作。\\n\'\n write_same(report_dir/\'interpretation_notes.txt\',memo.encode())\n paths=[config_dir/\'report_contract.json\',raw/\'all15_complete_benchmark.csv\',report_dir/\'interpretation_notes.txt\']+sorted(out.glob(\'*.csv\'))+[out/\'summary.json\']\n if self_tests is not None:\n  write_same(config_dir/\'self_tests.json\',json_bytes(self_tests));paths.append(config_dir/\'self_tests.json\')\n manifest={\'files\':[{\'name\':str(p.relative_to(root)),\'sha256\':sha(p)} for p in paths]};write_same(out/\'report_manifest.json\',json_bytes(manifest));paths.append(out/\'report_manifest.json\')\n bundle=root/\'results/summary\'/f\'{report_name}_review_bundle.zip\'\n with zipfile.ZipFile(bundle,\'w\',zipfile.ZIP_DEFLATED) as z:\n  for p in paths:z.write(p,str(p.relative_to(root)))\n  z.write(Path(__file__),\'src/task2_reporting.py\')\n return tables,summary,bundle\n', 'frozen_context_reproducibility': '"""Task 2.5: descriptive context dependence; biological directions remain blocked."""\nfrom pathlib import Path\nimport hashlib,json,zipfile,io,sys,subprocess\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.lines import Line2D\nfrom matplotlib.ticker import MaxNLocator\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nMETRICS=[\'mean_poisson_deviance\',\'normalized_mse\']\nCOLORS={\'both_better\':\'#207f88\',\'both_worse\':\'#c65e47\',\'sign_reversal\':\'#8b68a6\',\'near_zero_one_or_both\':\'#87918d\'}\ndef sha(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef js_bytes(o):return (json.dumps(o,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\').encode()\ndef immutable(path,data):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n if path.exists() and path.read_bytes()!=data:raise ValueError(\'Output changed; use a new analysis name: \'+str(path))\n if not path.exists():path.write_bytes(data)\ndef classify(x,y,eps):\n if not np.isfinite([x,y,eps]).all() or eps<0:raise ValueError(\'Invalid effect/threshold\')\n if abs(x)<=eps or abs(y)<=eps:return \'near_zero_one_or_both\'\n if x<0 and y<0:return \'both_better\'\n if x>0 and y>0:return \'both_worse\'\n return \'sign_reversal\'\ndef correlations(x,y):\n x=np.asarray(x,float);y=np.asarray(y,float);keep=np.isfinite(x)&np.isfinite(y);x=x[keep];y=y[keep]\n if len(x)<3 or np.ptp(x)==0 or np.ptp(y)==0:return len(x),np.nan,np.nan\n return len(x),float(np.corrcoef(x,y)[0,1]),float(pd.Series(x).rank().corr(pd.Series(y).rank()))\ndef load_report(path):\n with zipfile.ZipFile(path) as z:\n  if z.testzip() is not None or len(set(z.namelist()))!=len(z.namelist()):raise ValueError(\'Invalid ZIP\')\n  prefix=\'results/summary/task2_three_comparisons_v1/\'\n  manifest=json.loads(z.read(prefix+\'report_manifest.json\'))\n  for f in manifest[\'files\']:\n   if hashlib.sha256(z.read(f[\'name\'])).hexdigest()!=f[\'sha256\']:raise ValueError(\'Report manifest changed: \'+f[\'name\'])\n  cfg=json.loads(z.read(\'configs/task2_three_comparisons_v1/report_contract.json\'))\n  if hashlib.sha256(z.read(\'src/task2_reporting.py\')).hexdigest()!=cfg[\'report_engine_sha256\']:raise ValueError(\'Report source mismatch\')\n  summary=json.loads(z.read(prefix+\'summary.json\'))\n  if summary[\'status\']!=\'COMPLETE\' or summary[\'n_score_rows\']!=1290 or summary[\'scenario_B\']!=\'BLOCKED\':raise ValueError(\'Wrong source/scenario\')\n  effects=pd.read_csv(io.BytesIO(z.read(prefix+\'fixed_adaptive_effects.csv\')))\n  raw=pd.read_csv(io.BytesIO(z.read(\'results/raw/task2_three_comparisons_v1/all15_complete_benchmark.csv\')))\n  return effects,raw,cfg\n\ndef effect_points(effects,near_zero_percent=1e-5):\n required=[\'cell_line\',\'drug\',\'replicate\',\'comparison\',\'metric\',\'score\',\'reference_score\',\'delta_absolute\',\'delta_percent\']\n if effects[required].isna().any().any() or len(effects)!=120:raise ValueError(\'Complete finite effects required\')\n if set(effects.comparison)!={\'fixed\',\'adaptive\'} or set(effects.metric)!=set(METRICS):raise ValueError(\'Unexpected comparisons/metrics\')\n if not np.isfinite(effects[[\'score\',\'reference_score\',\'delta_absolute\',\'delta_percent\']].to_numpy(float)).all() or not effects.reference_score.gt(0).all():raise ValueError(\'Undefined relative effects\')\n if not np.allclose(effects.delta_absolute,effects.score-effects.reference_score,rtol=1e-10,atol=1e-14):raise ValueError(\'Delta subtraction mismatch\')\n if not np.allclose(effects.delta_percent,100*effects.delta_absolute/effects.reference_score,rtol=1e-10,atol=1e-12):raise ValueError(\'Percent denominator mismatch\')\n if not np.isfinite(near_zero_percent) or near_zero_percent<0:raise ValueError(\'Invalid band\')\n contexts=sorted(set(zip(effects.cell_line,effects.drug)));ids={c:i+1 for i,c in enumerate(contexts)}\n if len(contexts)!=15:raise ValueError(\'All 15 contexts required\')\n records=[]\n for (line,drug,metric,comparison),g in effects.groupby([\'cell_line\',\'drug\',\'metric\',\'comparison\'],sort=True):\n  if len(g)!=2 or set(g.replicate)!={\'replicate_1\',\'replicate_2\'}:raise ValueError(\'Expected two labels; no fabricated directions\')\n  a=g[g.replicate.eq(\'replicate_1\')].iloc[0];b=g[g.replicate.eq(\'replicate_2\')].iloc[0]\n  x=float(a.delta_percent);y=float(b.delta_percent)\n  records.append({\'context_id\':ids[(line,drug)],\'cell_line\':line,\'drug\':drug,\'metric\':metric,\'comparison\':comparison,\'delta_label_1\':a.delta_absolute,\'delta_label_2\':b.delta_absolute,\'percent_label_1\':x,\'percent_label_2\':y,\'reference_score_label_1\':a.reference_score,\'reference_score_label_2\':b.reference_score,\'score_label_1\':a.score,\'score_label_2\':b.score,\'raw_sign_class\':classify(x,y,0.),\'near_zero_percent\':near_zero_percent,\'band_class\':classify(x,y,near_zero_percent),\'absolute_percent_gap\':abs(x-y),\'direction_validity\':\'label sensitivity only; biological directions blocked\'})\n return pd.DataFrame(records)\n\ndef distribution_metrics(counts,label1=\'replicate_1\',label2=\'replicate_2\'):\n a=counts[counts.replicate.eq(label1)].set_index(\'category\').n_cells;b=counts[counts.replicate.eq(label2)].set_index(\'category\').n_cells\n labels=a.index.union(b.index);a=a.reindex(labels,fill_value=0).to_numpy(float);b=b.reindex(labels,fill_value=0).to_numpy(float)\n if min(a.sum(),b.sum())<=0:return np.nan,np.nan\n p=a/a.sum();q=b/b.sum();m=(p+q)/2\n def kl(x):\n  keep=x>0;return float(np.sum(x[keep]*np.log2(x[keep]/m[keep])))\n return float(.5*np.abs(p-q).sum()),float(.5*(kl(p)+kl(q)))\ndef composition(metadata,roster):\n counts=[];summary=[]\n wanted=roster[[\'cell_line\',\'drug\',\'replicate\',\'n_cells\']].drop_duplicates()\n if wanted.duplicated([\'cell_line\',\'drug\',\'replicate\']).any():raise ValueError(\'Conflicting roster\')\n if len(wanted)!=30:raise ValueError(\'Full composition roster required\')\n if metadata is None:\n  return pd.DataFrame(columns=[\'cell_line\',\'drug\',\'replicate\',\'field\',\'category\',\'n_cells\',\'fraction\']),pd.DataFrame([{\'cell_line\':l,\'drug\':d,\'field\':f,\'status\':\'METADATA_UNAVAILABLE\',\'TV_all\':np.nan,\'JSD_all_bits\':np.nan,\'TV_known\':np.nan,\'JSD_known_bits\':np.nan} for l,d in wanted[[\'cell_line\',\'drug\']].drop_duplicates().itertuples(index=False,name=None) for f in [\'dose_raw\',\'sgRNA_raw\']])\n if \'evaluation_eligible\' not in metadata:raise ValueError(\'Metadata eligibility missing\')\n active=metadata.loc[metadata.evaluation_eligible.fillna(False).to_numpy(dtype=bool)].copy()\n if \'matrix_row\' not in active or active.matrix_row.isna().any() or active.matrix_row.duplicated().any():raise ValueError(\'Metadata unique matrix_row required\')\n actual=active.groupby([\'cell_line\',\'drug\',\'replicate\']).size()\n for r in wanted.itertuples(index=False):\n  if actual.get((r.cell_line,r.drug,r.replicate),0)!=r.n_cells:raise ValueError(\'Composition cells differ from score common-I\')\n if len(active)!=int(wanted.n_cells.sum()):raise ValueError(\'Unexpected eligible metadata contexts\')\n for field in [\'dose_raw\',\'sgRNA_raw\']:\n  if field not in active:\n   summary.extend({\'cell_line\':l,\'drug\':d,\'field\':field,\'status\':\'FIELD_UNAVAILABLE\',\'TV_all\':np.nan,\'JSD_all_bits\':np.nan,\'TV_known\':np.nan,\'JSD_known_bits\':np.nan} for l,d in wanted[[\'cell_line\',\'drug\']].drop_duplicates().itertuples(index=False,name=None));continue\n  categories=active[field].astype(\'string\').str.strip()\n  categories=categories.mask(categories.isin([\'\',\'NA\',\'NaN\',\'nan\',\'None\',\'<NA>\']))\n  if field==\'dose_raw\':\n   numeric=pd.to_numeric(categories,errors=\'coerce\');known=numeric.notna()&np.isfinite(numeric)\n   categories.loc[known]=numeric.loc[known].map(lambda v:format(float(v),\'.12g\'))\n  active[\'category\']=categories.fillna(\'<missing>\')\n  table=active.groupby([\'cell_line\',\'drug\',\'replicate\',\'category\'],dropna=False).size().reset_index(name=\'n_cells\');table[\'field\']=field\n  table[\'fraction\']=table.n_cells/table.groupby([\'cell_line\',\'drug\',\'replicate\']).n_cells.transform(\'sum\');counts.extend(table.to_dict(\'records\'))\n  for (line,drug),g in table.groupby([\'cell_line\',\'drug\']):\n   known=g[g.category.ne(\'<missing>\')];tv,js=distribution_metrics(g);ktv,kjs=distribution_metrics(known)\n   fractions={rep:float(g.loc[g.replicate.eq(rep)&g.category.eq(\'<missing>\'),\'fraction\'].sum()) for rep in [\'replicate_1\',\'replicate_2\']}\n   status=\'ALL_MISSING\' if known.empty else \'ONE_LABEL_NO_KNOWN_VALUES\' if np.isnan(ktv) else \'CONSTANT_KNOWN_COMPOSITION\' if known.category.nunique()==1 else \'DESCRIPTIVE_COMPARISON_AVAILABLE\'\n   if status==\'ALL_MISSING\':tv=js=np.nan\n   units=\'NOT_RECORDED\'\n   if field==\'dose_raw\' and \'dose_unit\' in active:units=\' | \'.join(sorted(active.loc[active.cell_line.eq(line)&active.drug.eq(drug),\'dose_unit\'].dropna().astype(str).unique())) or \'NOT_RECORDED\'\n   summary.append({\'cell_line\':line,\'drug\':drug,\'field\':field,\'status\':status,\'TV_all\':tv,\'JSD_all_bits\':js,\'TV_known\':ktv,\'JSD_known_bits\':kjs,\'missing_fraction_label_1\':fractions[\'replicate_1\'],\'missing_fraction_label_2\':fractions[\'replicate_2\'],\'known_categories\':int(known.category.nunique()),\'dose_units\':units})\n return pd.DataFrame(counts),pd.DataFrame(summary)\n\ndef make_scatter(points,metric,scale,out):\n fig,axs=plt.subplots(1,3,figsize=(14,6),gridspec_kw={\'width_ratios\':[1,1,.72]},layout=\'constrained\')\n labels=[]\n for ax,comparison in zip(axs[:2],[\'fixed\',\'adaptive\']):\n  g=points[points.metric.eq(metric)&points.comparison.eq(comparison)].sort_values(\'context_id\');cols=[\'percent_label_1\',\'percent_label_2\'] if scale==\'percent\' else [\'delta_label_1\',\'delta_label_2\']\n  values=g[cols].to_numpy(float);lo=min(float(values.min()),0.);hi=max(float(values.max()),0.);span=hi-lo\n  if span==0:span=max(float(g.near_zero_percent.iloc[0])*4,1e-5) if scale==\'percent\' else 1e-6\n  lo-=span*.2;hi+=span*.2\n  if scale==\'percent\':\n   band=float(g.near_zero_percent.iloc[0]);ax.axvspan(-band,band,color=\'#dfe5e1\',alpha=.7);ax.axhspan(-band,band,color=\'#dfe5e1\',alpha=.7)\n  ax.axhline(0,color=\'#5b6663\',linewidth=.9);ax.axvline(0,color=\'#5b6663\',linewidth=.9);ax.plot([lo,hi],[lo,hi],linestyle=\'--\',color=\'#a6b0aa\',linewidth=.8)\n  for r in g.itertuples():\n   x=getattr(r,cols[0]);y=getattr(r,cols[1]);ax.scatter(x,y,c=COLORS[r.band_class],s=40,edgecolors=\'white\',linewidth=.6,zorder=3)\n   labels.append((ax,r.context_id,x,y))\n  ax.set(xlim=(lo,hi),ylim=(lo,hi),xlabel=\'replicate_1 label\',ylabel=\'replicate_2 label\',title=\'M1 - M0\' if comparison==\'fixed\' else \'M2 - M1\');ax.set_aspect(\'equal\',adjustable=\'box\');ax.xaxis.set_major_locator(MaxNLocator(nbins=5));ax.yaxis.set_major_locator(MaxNLocator(nbins=5));ax.ticklabel_format(axis=\'both\',style=\'sci\',scilimits=(-3,3));ax.grid(alpha=.15)\n axs[2].axis(\'off\');key=points[[\'context_id\',\'cell_line\',\'drug\']].drop_duplicates().sort_values(\'context_id\');axs[2].text(0,1,\'CONTEXT KEY\\n\\n\'+\'\\n\'.join(f\'{r.context_id:2d}  {r.cell_line} / {r.drug}\' for r in key.itertuples()),va=\'top\',fontsize=9)\n handles=[Line2D([0],[0],marker=\'o\',color=\'none\',markerfacecolor=c,label=k.replace(\'_\',\' \'),markersize=6) for k,c in COLORS.items()];axs[2].legend(handles=handles,loc=\'lower left\',frameon=False,fontsize=8)\n unit=\'relative difference (%)\' if scale==\'percent\' else \'absolute loss difference\';fig.suptitle(f\'{metric}: {unit}\\nNegative = lower loss; label sensitivity, NOT biological replicate directions\',fontsize=12)\n fig.canvas.draw();fig.set_layout_engine(None)\n # Deterministic collision avoidance: labels move; saved data points never move.\n occupied={ax:[] for ax in axs[:2]};scale_px=fig.dpi/72\n for ax,identifier,x,y in labels:\n  px,py=ax.transData.transform((x,y));width=(6*len(str(identifier))+4)*scale_px;height=12*scale_px;chosen=None\n  for radius in [12,20,30,42,56,72,90]:\n   for angle in np.linspace(0,2*np.pi,16,endpoint=False):\n    dx=radius*np.cos(angle)*scale_px;dy=radius*np.sin(angle)*scale_px;rect=(px+dx-width/2,py+dy-height/2,px+dx+width/2,py+dy+height/2)\n    inside=rect[0]>ax.bbox.x0+2 and rect[1]>ax.bbox.y0+2 and rect[2]<ax.bbox.x1-2 and rect[3]<ax.bbox.y1-2\n    overlaps=any(not (rect[2]<r[0] or rect[0]>r[2] or rect[3]<r[1] or rect[1]>r[3]) for r in occupied[ax])\n    if inside and not overlaps:chosen=(dx/scale_px,dy/scale_px,rect);break\n   if chosen is not None:break\n  if chosen is None:raise RuntimeError(\'Could not place every context label; increase figure size\')\n  dx,dy,rect=chosen;occupied[ax].append(rect);ax.annotate(str(identifier),(x,y),xytext=(dx,dy),textcoords=\'offset points\',fontsize=8,ha=\'center\',va=\'center\',arrowprops={\'arrowstyle\':\'-\',\'color\':\'#aeb7b1\',\'lw\':.5})\n out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)\n fig.savefig(out.with_suffix(\'.png\'),dpi=180)\n with matplotlib.rc_context({\'svg.hashsalt\':ENGINE_SHA}):fig.savefig(out.with_suffix(\'.svg\'),metadata={\'Date\':None})\n plt.close(fig)\n\ndef run_analysis(root,input_zip,metadata_path=None,near_zero_percent=1e-5,bands=(0.,1e-5,.001,.01,.1),name=\'task2_context_reproducibility_v1\',self_tests=None):\n root=Path(root).resolve();input_zip=Path(input_zip).resolve()\n if Path(name).name!=name or name in [\'\', \'.\', \'..\']:raise ValueError(\'Invalid name\')\n if self_tests is not None and self_tests.get(\'status\')!=\'PASS\':raise ValueError(\'Self tests must pass\')\n effects,raw,source_cfg=load_report(input_zip);points=effect_points(effects,near_zero_percent)\n roster=raw[[\'cell_line\',\'drug\',\'replicate\',\'n_cells\']].drop_duplicates();metadata=None;metadata_identity={\'status\':\'NOT_AVAILABLE\'}\n if metadata_path is not None and Path(metadata_path).is_file():\n  metadata_path=Path(metadata_path).resolve();run=root/\'data/splits/step2_q05_seed20261008_v1\';pair=json.loads((run/\'A_preprocessing/evaluation_pair_contract.json\').read_text());metric=json.loads((run/\'evaluation/metric_contract.json\').read_text())\n  if sha(metadata_path)!=pair[\'cells_sha256\'] or pair[\'cells_sha256\']!=metric[\'cells_sha256\']:raise ValueError(\'Metadata hash does not match frozen evaluation cells\')\n  source_zip=root/\'results/summary/task2_step3_review_bundle_task2_ordered_controls_num_v3.zip\'\n  if sha(source_zip)!=source_cfg[\'input_zip_sha256\']:raise ValueError(\'Source 2.3 ZIP mismatch\')\n  with zipfile.ZipFile(source_zip) as z:\n   contract=json.loads(z.read(\'experiment_contract.json\'))\n  if sha(run/\'evaluation/metric_contract.json\')!=contract[\'metric_sha256\']:raise ValueError(\'Metric metadata linkage changed\')\n  metadata=pd.read_parquet(metadata_path)\n  if sha(metadata_path)!=pair[\'cells_sha256\']:raise ValueError(\'Metadata changed during read\')\n  metadata_identity={\'status\':\'VERIFIED_EVALUATION_METADATA\',\'sha256\':sha(metadata_path),\'columns\':list(metadata.columns)}\n counts,comp=composition(metadata,roster);sensitivity=[];repro=[];assoc=[]\n for band in bands:\n  if not np.isfinite(band) or band<0:raise ValueError(\'Invalid sensitivity band\')\n  for (metric,comparison),g in points.groupby([\'metric\',\'comparison\']):\n   classes=[classify(x,y,band) for x,y in zip(g.percent_label_1,g.percent_label_2)]\n   for key in COLORS:sensitivity.append({\'metric\':metric,\'comparison\':comparison,\'near_zero_percent\':band,\'class\':key,\'n_contexts\':classes.count(key)})\n for (metric,comparison),g in points.groupby([\'metric\',\'comparison\']):\n  n,p,s=correlations(g.percent_label_1,g.percent_label_2);repro.append({\'metric\':metric,\'comparison\':comparison,\'n_contexts\':n,\'pearson_descriptive\':p,\'spearman_descriptive\':s,\'inference\':\'descriptive; labels share data source\'})\n  for field in [\'dose_raw\',\'sgRNA_raw\']:\n   merged=g.merge(comp[comp.field.eq(field)],on=[\'cell_line\',\'drug\'],how=\'left\');n,p,s=correlations(merged.TV_known,merged.absolute_percent_gap)\n   assoc.append({\'metric\':metric,\'comparison\':comparison,\'field\':field,\'n_valid_contexts\':n,\'pearson_TV_vs_effect_gap\':p,\'spearman_TV_vs_effect_gap\':s,\'interpretation\':\'association only; no dose/sgRNA-stratified saved losses, cannot establish explanation or causal attribution\'})\n config={\'kind\':\'task2-step5-label-sensitivity-v1\',\'input_report_sha256\':sha(input_zip),\'source_fit_zip_sha256\':source_cfg[\'input_zip_sha256\'],\'engine_sha256\':ENGINE_SHA,\'metadata\':metadata_identity,\'near_zero_percent\':near_zero_percent,\'numeric_band_basis\':\'1e-5 percent equals 1e-7 relative deviance gate; numerical resolution band, NOT practical biological threshold\',\'sensitivity_bands_percent\':list(bands),\'direction_mode\':\'Scenario A labels; legitimate biological directions BLOCKED\',\'seed\':\'no resampling; saved inputs only\',\'composition\':\'categorical dose/sgRNA fractions in verified eligible common-I; TV/JSD descriptive, missingness explicit; no cell bootstrap/p-values\',\'units\':\'within context raw dose categories; units recorded when available\',\'environment\':{\'python\':sys.version,\'numpy\':np.__version__,\'pandas\':pd.__version__,\'matplotlib\':matplotlib.__version__}}\n out=root/\'results/summary\'/name;figs=root/\'figures\'/name;cfg=root/\'configs\'/name\n try:config[\'git_commit\']=subprocess.run([\'git\',\'rev-parse\',\'--verify\',\'HEAD\'],cwd=root,check=True,capture_output=True,text=True).stdout.strip()\n except (OSError,subprocess.CalledProcessError):config[\'git_commit\']=None\n immutable(cfg/\'analysis_contract.json\',js_bytes(config))\n tables={\'context_effect_points\':points,\'near_zero_sensitivity\':pd.DataFrame(sensitivity),\'label_correlations\':pd.DataFrame(repro),\'composition_counts\':counts,\'composition_differences\':comp,\'composition_effect_associations\':pd.DataFrame(assoc)}\n for key,df in tables.items():immutable(out/(key+\'.csv\'),df.to_csv(index=False).encode())\n for metric in METRICS:\n  for scale in [\'percent\',\'absolute\']:make_scatter(points,metric,scale,figs/f\'{metric}_{scale}_scatter\')\n summary={\'status\':\'COMPLETE_LABEL_SENSITIVITY\',\'genuine_biological_direction_analysis\':\'BLOCKED_PENDING_PROVENANCE\',\'n_contexts\':15,\'n_effect_points\':len(points),\'metadata_status\':metadata_identity[\'status\'],\'composition_status_counts\':comp.status.value_counts().to_dict(),\'raw_B_access\':False,\'model_refit\':False}\n immutable(out/\'summary.json\',js_bytes(summary))\n notes=\'任务2.5：全部15 contexts的fixed/adaptive效应散点图，分别保存绝对差值及相对百分比，normalized MSE为次指标。负差值表示改善。\\n当前仅为replicate-label sensitivity；两个合法biological directions仍blocked，也不把labels当统计独立复制。\\nnear-zero主带来自原数值gate，不是biological意义阈值；同时完整报告预声明阈值敏感性。\\nDose/sgRNA composition仅用哈希验证的评价metadata，不读取表达或B。缺失字段/all-missing明确记录，dose单位不完整时只作原始类别比较。TV/JSD与效应gap关联为探索性描述，不能证明composition解释了效应差异；缺少按dose/sgRNA分层的已保存loss，未实施调整后效应或因果估计。\\n不做cell bootstrap、p值、confidence intervals或选择性删除contexts。独立biological samples不足，无法由当前数据推断普遍生物学规律。\\n\'\n note=root/\'reports\'/name/\'interpretation_notes.txt\';immutable(note,notes.encode())\n files=[cfg/\'analysis_contract.json\',note,out/\'summary.json\']+sorted(out.glob(\'*.csv\'))+sorted(figs.glob(\'*\'))\n if self_tests is not None:\n  immutable(cfg/\'self_tests.json\',js_bytes(self_tests));files.append(cfg/\'self_tests.json\')\n manifest={\'files\':[{\'name\':str(p.relative_to(root)),\'sha256\':sha(p)} for p in files]};manifest_path=out/\'review_manifest.json\'\n immutable(manifest_path,js_bytes(manifest));files.append(manifest_path)\n bundle=root/\'results/summary\'/f\'{name}_review_bundle.zip\'\n with zipfile.ZipFile(bundle,\'w\',zipfile.ZIP_DEFLATED) as z:\n  for p in files:z.write(p,str(p.relative_to(root)))\n  z.write(Path(__file__),\'src/context_reproducibility.py\')\n return tables,summary,bundle\n', 'benchmark_figures': '"""Task2.6: figures and complete table from saved, hash-linked results only."""\nfrom pathlib import Path\nimport json,io,zipfile,hashlib,sys,subprocess\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.ticker import MaxNLocator\nimport frozen_task2_reporting as rp\nimport frozen_context_reproducibility as cr\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nMETRICS=rp.METRICS\nREPRESENTATIVES=[(\'A172\',\'Lapatinib\'),(\'A172\',\'Trametinib\'),(\'T98G\',\'Nintedanib\'),(\'U87MG\',\'ZSTK474\')]\nMETHOD_COLORS={\'M0\':\'#687b78\',\'M1\':\'#c07847\',\'M2\':\'#217f91\',\'M3\':\'#936b9b\',\'M4\':\'#8a9768\'}\n\ndef zip_verified(path,manifest_name):\n with zipfile.ZipFile(path) as z:\n  if z.testzip() is not None or len(set(z.namelist()))!=len(z.namelist()):raise ValueError(\'Invalid ZIP\')\n  manifest=json.loads(z.read(manifest_name))\n  for f in manifest[\'files\']:\n   if hashlib.sha256(z.read(f[\'name\'])).hexdigest()!=f[\'sha256\']:raise ValueError(\'Manifest mismatch: \'+f[\'name\'])\n\ndef load_inputs(fit_zip,report_zip,context_zip):\n df,fit_contract,grid,failed,candidates,edges=rp.read_verified_bundle(fit_zip)\n zip_verified(report_zip,\'results/summary/task2_three_comparisons_v1/report_manifest.json\')\n zip_verified(context_zip,\'results/summary/task2_context_reproducibility_v1/review_manifest.json\')\n with zipfile.ZipFile(report_zip) as z:\n  cfg=json.loads(z.read(\'configs/task2_three_comparisons_v1/report_contract.json\'))\n  if cfg[\'input_zip_sha256\']!=rp.sha(fit_zip) or cfg[\'report_engine_sha256\']!=rp.ENGINE_SHA:raise ValueError(\'Step2.4 lineage/source mismatch\')\n  effects=pd.read_csv(io.BytesIO(z.read(\'results/summary/task2_three_comparisons_v1/fixed_adaptive_effects.csv\')))\n with zipfile.ZipFile(context_zip) as z:\n  ctx=json.loads(z.read(\'configs/task2_context_reproducibility_v1/analysis_contract.json\'))\n  if ctx[\'input_report_sha256\']!=rp.sha(report_zip) or ctx[\'source_fit_zip_sha256\']!=rp.sha(fit_zip) or ctx[\'engine_sha256\']!=cr.ENGINE_SHA:raise ValueError(\'Step2.5 lineage/source mismatch\')\n  points=pd.read_csv(io.BytesIO(z.read(\'results/summary/task2_context_reproducibility_v1/context_effect_points.csv\')))\n  expected=cr.effect_points(effects,ctx[\'near_zero_percent\']);pd.testing.assert_frame_equal(points,expected,check_dtype=False,rtol=1e-10,atol=1e-13)\n # Stored audit eigenvalues only; never load model/count NPZ or recompute graphs.\n values=[]\n with zipfile.ZipFile(fit_zip) as z:\n  for r in df.itertuples():\n   tag=r.method if pd.isna(r.seed_index) else f\'{r.method}_seed{int(r.seed_index):03d}\'\n   audit=json.loads(z.read(f\'scopes/scope_{r.scope_id:02d}/outer/models/{tag}/graph_audit.json\'));gene=audit[\'effective_gene\']\n   if not np.isclose(gene[\'trace_Lg\'],r.trace_Lg,rtol=1e-10,atol=1e-12) or gene[\'zero_weight_edges\']!=r.zero_weight_edges or not np.isclose(gene[\'weight_sum\'],r.weight_sum):raise ValueError(\'Graph audit/score mismatch\')\n   values.append({\'support_edges\':gene[\'support_edges\'],\'effective_gene_lambda_max\':gene[\'lambda_max\'],\'beta_times_gene_lambda_max\':r.beta*gene[\'lambda_max\'],\'zero_weight_edge_fraction\':gene[\'zero_weight_edges\']/gene[\'support_edges\'] if gene[\'support_edges\'] else np.nan,\'gene_operator_active\':r.beta>0})\n df=pd.concat([df.reset_index(drop=True),pd.DataFrame(values)],axis=1)\n return df,points,fit_contract,grid,failed,candidates,ctx\n\ndef calculate_tables(df):\n rp.validate_frame(df);table=[];effects=[];pairs=[];null=[]\n for sid,g in df.groupby(\'scope_id\'):\n  meta=g[rp.META].iloc[0].to_dict();base={m:g[g.method.eq(m)].iloc[0] for m in [\'M0\',\'M1\',\'M2\']}\n  for metric in METRICS:\n   for method,m in g.groupby(\'method\'):\n    x=m[metric].to_numpy(float);n=len(x);ref=float(base[\'M0\'][metric]);mean=float(x.mean())\n    table.append({**meta,\'metric\':metric,\'method\':method,\'score_mean\':mean,\'score_min\':float(x.min()),\'score_max\':float(x.max()),\'seed_q05\':float(np.quantile(x,.05)) if n>1 else np.nan,\'seed_q95\':float(np.quantile(x,.95)) if n>1 else np.nan,\'delta_vs_M0_absolute\':mean-ref,\'delta_vs_M0_percent\':rp.relative(mean-ref,ref),\'n_models\':n,\'n_random_seeds\':n if method in [\'M3\',\'M4\'] else 0,\'uncertainty_summary\':\'descriptive saved-seed range, NOT biological CI\' if n>1 else \'one deterministic model; no biological CI available\',\'biological_CI\':\'NOT_ESTIMABLE\',\'alpha_min\':float(m.alpha.min()),\'alpha_max\':float(m.alpha.max()),\'beta_min\':float(m.beta.min()),\'beta_max\':float(m.beta.max()),\'lambda_rules\':\' | \'.join(sorted(m.lambda_rule.unique())),\'conditional_EDF_mean\':float(m.conditional_trace_H.mean()),\'smoothing_scale_mean\':float(m.beta_times_gene_lambda_max.mean()),\'zero_weight_edge_fraction_mean\':float(m.zero_weight_edge_fraction.mean()),\'weight_sum_mean\':float(m.weight_sum.mean()),\'scope_interpretation\':\'Scenario A replicate label; biological directions BLOCKED\'})\n   for method in [\'M0\',\'M1\',\'M2\']:\n    x=float(base[method][metric]);ref=float(base[\'M0\'][metric]);effects.append({**meta,\'metric\':metric,\'method\':method,\'loss\':x,\'delta_vs_M0\':x-ref,\'percent_vs_M0\':rp.relative(x-ref,ref)})\n   for r in g[g.method.eq(\'M3\')].itertuples():\n    x=float(base[\'M2\'][metric]);y=float(getattr(r,metric));pairs.append({**meta,\'metric\':metric,\'seed_index\':int(r.seed_index),\'M2_score\':x,\'M3_score\':y,\'delta_M2_minus_M3\':x-y,\'percent_M2_minus_M3\':rp.relative(x-y,y)})\n   for r in g[g.method.eq(\'M4\')].itertuples():\n    obs=float(base[\'M1\'][metric]);value=float(getattr(r,metric));null.append({**meta,\'metric\':metric,\'seed_index\':int(r.seed_index),\'M1_score\':obs,\'M4_score\':value,\'plot_percent_M4_minus_M1\':rp.relative(value-obs,obs),\'M1_alpha\':base[\'M1\'].alpha,\'M1_beta\':base[\'M1\'].beta,\'M4_alpha\':r.alpha,\'M4_beta\':r.beta})\n return {\'Table1_complete_benchmark\':pd.DataFrame(table),\'Figure1_all_context_effects\':pd.DataFrame(effects),\'Figure3_paired_seed_effects\':pd.DataFrame(pairs),\'Figure2_all_null_raw\':pd.DataFrame(null),\'all_method_seed_scores_and_audits\':df}\n\ndef finish(fig,path):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n fig.savefig(path.with_suffix(\'.png\'),dpi=180)\n with matplotlib.rc_context({\'svg.hashsalt\':ENGINE_SHA}):fig.savefig(path.with_suffix(\'.svg\'),metadata={\'Date\':None})\n plt.close(fig)\ndef axes_style(ax):\n ax.grid(alpha=.15);ax.xaxis.set_major_locator(MaxNLocator(nbins=5));ax.ticklabel_format(axis=\'x\',style=\'sci\',scilimits=(-3,3))\ndef context_key(frame):return frame[[\'cell_line\',\'drug\']].drop_duplicates().sort_values([\'cell_line\',\'drug\']).reset_index(drop=True)\ndef figure1(csv_path,metric,out):\n data=pd.read_csv(csv_path);data=data[data.metric.eq(metric)];key=context_key(data);mapping={(r.cell_line,r.drug):i for i,r in key.iterrows()}\n fig,axs=plt.subplots(2,2,figsize=(14,11),layout=\'constrained\',sharey=True)\n for col,rep in enumerate([\'replicate_1\',\'replicate_2\']):\n  g=data[data.replicate.eq(rep)]\n  for method,offset in [(\'M0\',-.18),(\'M1\',0),(\'M2\',.18)]:\n   m=g[g.method.eq(method)];y=np.array([mapping[(r.cell_line,r.drug)] for r in m.itertuples()])+offset\n   axs[0,col].scatter(m.loss,y,s=26,c=METHOD_COLORS[method],label=method)\n   if method!=\'M0\':axs[1,col].scatter(m.percent_vs_M0,y,s=26,c=METHOD_COLORS[method],label=method)\n  for row in [0,1]:\n   ax=axs[row,col];ax.set_yticks(range(15),[f\'{r.cell_line} / {r.drug}\' for r in key.itertuples()],fontsize=8);ax.invert_yaxis() if col==row==0 else None;axes_style(ax);ax.legend(loc=\'upper left\',bbox_to_anchor=(1.01,1),fontsize=8,frameon=False);ax.set_title(rep+\' label\')\n  axs[0,col].set_xlabel(\'Held-out loss\');axs[1,col].axvline(0,color=\'#526762\',lw=.8);axs[1,col].set_xlabel(\'(method - M0) / M0 (%) ; negative = lower loss\')\n fig.suptitle(f\'Figure 1 | {metric}\\nAll 15 contexts; Scenario A labels, not biological directions\',fontsize=13);finish(fig,out)\ndef figure2(csv_path,metric,contexts,out):\n data=pd.read_csv(csv_path);data=data[data.metric.eq(metric)]\n fig,axs=plt.subplots(len(contexts),2,figsize=(14,2.8*len(contexts)+1.1),layout=\'constrained\',squeeze=False)\n for i,(line,drug) in enumerate(contexts):\n  for j,rep in enumerate([\'replicate_1\',\'replicate_2\']):\n   ax=axs[i,j];g=data[data.cell_line.eq(line)&data.drug.eq(drug)&data.replicate.eq(rep)].sort_values(\'seed_index\');x=g.plot_percent_M4_minus_M1.to_numpy(float)\n   if len(g)!=20 or not np.isfinite(x).all():raise ValueError(\'Figure2 requires all 20 finite null values\')\n   ax.boxplot([x],vert=False,positions=[0],widths=.18,whis=(0,100),showfliers=False,patch_artist=True,boxprops={\'facecolor\':\'#e2e9df\',\'edgecolor\':\'#819576\'},medianprops={\'color\':\'#42594a\'})\n   ax.scatter(x,.12*np.sin(g.seed_index.to_numpy()*2.399963),s=18,c=METHOD_COLORS[\'M4\'],zorder=3)\n   ax.axvline(0,c=METHOD_COLORS[\'M1\'],lw=1.4,label=\'real M1\');ax.set_yticks([]);ax.set_ylim(-.28,.28);axes_style(ax)\n   if np.ptp(x)==0 and float(x[0])==0:ax.set_xlim(-1e-5,1e-5)\n   r=g.iloc[0];ax.set_title(f\'{line} / {drug} | {rep}\',fontsize=10)\n   ax.set_xlabel(\'M4 - M1 (% of M1); >0 favors real PPI\',fontsize=8)\n   note=f"M1 D={r.M1_score:.6g}; alpha={r.M1_alpha:.4g}, beta={r.M1_beta:.4g}\\nM4 alpha=[{g.M4_alpha.min():.3g},{g.M4_alpha.max():.3g}], beta=[{g.M4_beta.min():.3g},{g.M4_beta.max():.3g}]"\n   ax.text(.02,.98,note,transform=ax.transAxes,va=\'top\',fontsize=7)\n fig.suptitle(f\'Figure 2 | {metric}: PPI null comparison\\nM1 and every M4 seed independently use identical frozen T/V coarse+fine grids; 20 seeds, no biological CI\',fontsize=11);finish(fig,out)\ndef figure3(pair_csv,audit_csv,metric,out):\n pairs=pd.read_csv(pair_csv);pairs=pairs[pairs.metric.eq(metric)];audits=pd.read_csv(audit_csv);key=context_key(pairs);mapping={(r.cell_line,r.drug):i for i,r in key.iterrows()}\n fig,axs=plt.subplots(3,2,figsize=(15,15),layout=\'constrained\');axes=axs.ravel()\n labels=[\'Paired M2 - M3 (% of each M3); negative favors M2\',\'Zero-weight edge fraction\',\'Total edge weight\',\'Effective gene scale: beta * lambda_max(Lg)\',\'Conditional EDF: trace(H)\']\n for rep,offset,color in [(\'replicate_1\',-.20,\'#217f91\'),(\'replicate_2\',.20,\'#c07847\')]:\n  for (line,drug),g in pairs[pairs.replicate.eq(rep)].groupby([\'cell_line\',\'drug\']):\n   y=mapping[(line,drug)]+offset;x=g.percent_M2_minus_M3.to_numpy(float);axes[0].plot(np.quantile(x,[.05,.95]),[y,y],c=color,lw=1);axes[0].scatter(x.mean(),y,c=color,s=25)\n   a=audits[audits.cell_line.eq(line)&audits.drug.eq(drug)&audits.replicate.eq(rep)];m2=a[a.method.eq(\'M2\')].iloc[0];m3=a[a.method.eq(\'M3\')]\n   for axis,column in [(axes[1],\'zero_weight_edge_fraction\'),(axes[2],\'weight_sum\'),(axes[3],\'beta_times_gene_lambda_max\'),(axes[4],\'conditional_trace_H\')]:\n    x=m3[column].to_numpy(float);axis.plot(np.quantile(x,[.05,.95]),[y+.10,y+.10],c=color,lw=1);axis.scatter(x.mean(),y+.10,c=color,s=24,marker=\'D\');axis.scatter(m2[column],y-.10,c=color,s=24,marker=\'o\')\n for ax,label in zip(axes[:5],labels):\n  ax.set_yticks(range(15),[f\'{r.cell_line} / {r.drug}\' for r in key.itertuples()],fontsize=7);ax.invert_yaxis();ax.set_xlabel(label,fontsize=9);axes_style(ax)\n axes[0].axvline(0,c=\'#526762\',lw=.8);axes[1].set_xlim(-.02,1.02);axes[5].axis(\'off\');axes[5].text(0,1,\'READING THIS FIGURE\\n\\nBlue: replicate_1 label\\nOrange: replicate_2 label\\n\\nCircle: saved M2 model\\nDiamond: mean of 20 M3 models\\nBars: saved seed 5/95% quantiles\\nNOT biological confidence intervals\\n\\nMatched controls preserve:\\n- weight multiset including zeros\\n- total weight and trace(Lg)\\n\\nThey do NOT preserve:\\n- lambda_max / spectral shape\\n- conditional EDF\\n\\nConditional EDF is not total adaptive DF.\\nZero weights are not physically removed\\nfrom the recorded support edge list.\',va=\'top\',fontsize=10)\n fig.suptitle(f\'Figure 3 | {metric}: adaptive vs matched attenuation\\nAll contexts; paired effects and stored smoothing audits\',fontsize=13);finish(fig,out)\n\ndef run_figures(root,fit_zip,report_zip,context_zip,name=\'task2_four_figures_table1_v1\',self_tests=None):\n root=Path(root).resolve()\n if Path(name).name!=name or name in [\'\', \'.\', \'..\']:raise ValueError(\'Invalid name\')\n if self_tests is not None and self_tests.get(\'status\')!=\'PASS\':raise ValueError(\'Self tests must pass\')\n df,points,fit_contract,grid,failed,candidates,ctx=load_inputs(fit_zip,report_zip,context_zip);tables=calculate_tables(df)\n config={\'kind\':\'task2-step6-figures-table-v1\',\'inputs\':{str(Path(p).name):rp.sha(p) for p in [fit_zip,report_zip,context_zip]},\'engine_sha256\':ENGINE_SHA,\'report_helper_sha256\':rp.ENGINE_SHA,\'context_helper_sha256\':cr.ENGINE_SHA,\'representative_contexts_predeclared\':REPRESENTATIVES,\'seed_count\':20,\'fit_rules\':\'same frozen independent T/V coarse+fine search for M1 and each M4; all parameters included in raw table\',\'gene_smoothing_scale\':\'beta times stored lambda_max of effective Lg; not recomputed or retuned\',\'Table1_percent\':\'(mean saved loss - M0 loss) / M0 *100\',\'Figure3_percent\':\'mean of paired seed percentages, each denominator M3_seed\',\'uncertainty\':\'randomized seed quantiles only; biological CI unavailable; no p-values/cell bootstrap\',\'direction_mode\':\'Scenario A replicate labels; genuine biological directions BLOCKED\',\'environment\':{\'python\':sys.version,\'numpy\':np.__version__,\'pandas\':pd.__version__,\'matplotlib\':matplotlib.__version__}}\n try:config[\'git_commit\']=subprocess.run([\'git\',\'rev-parse\',\'--verify\',\'HEAD\'],cwd=root,check=True,capture_output=True,text=True).stdout.strip()\n except (OSError,subprocess.CalledProcessError):config[\'git_commit\']=None\n out=root/\'results/summary\'/name;raw=root/\'results/raw\'/name;figs=root/\'figures\'/name;cfg=root/\'configs\'/name;reports=root/\'reports\'/name\n rp.write_same(cfg/\'figure_contract.json\',rp.json_bytes(config))\n rp.write_same(cfg/\'source_effective_parameter_grid.json\',rp.json_bytes(grid))\n for key,table in tables.items():rp.write_same(raw/(key+\'.csv\'),table.to_csv(index=False).encode())\n rp.write_same(raw/\'Figure4_context_effect_points.csv\',points.to_csv(index=False).encode())\n rp.write_same(out/\'failed_candidates.csv\',failed.to_csv(index=False).encode());rp.write_same(out/\'candidate_status_summary.csv\',candidates.to_csv(index=False).encode())\n if self_tests is not None:rp.write_same(cfg/\'self_tests.json\',rp.json_bytes(self_tests))\n contexts=list(context_key(df).itertuples(index=False,name=None))\n for metric in METRICS:\n  suffix=\'primary\' if metric==METRICS[0] else \'secondary\'\n  figure1(raw/\'Figure1_all_context_effects.csv\',metric,figs/f\'Figure1_all_context_effects_{suffix}\')\n  figure2(raw/\'Figure2_all_null_raw.csv\',metric,REPRESENTATIVES,figs/f\'Figure2_PPI_identity_{suffix}\')\n  figure3(raw/\'Figure3_paired_seed_effects.csv\',raw/\'all_method_seed_scores_and_audits.csv\',metric,figs/f\'Figure3_adaptive_attenuation_{suffix}\')\n  cr.make_scatter(pd.read_csv(raw/\'Figure4_context_effect_points.csv\'),metric,\'percent\',figs/f\'Figure4_label_consistency_{suffix}\')\n  if suffix==\'primary\':\n   for i in range(3):figure2(raw/\'Figure2_all_null_raw.csv\',metric,contexts[i*5:(i+1)*5],figs/f\'Figure2_all_contexts_supplement_{i+1:02d}\')\n summary={\'status\':\'COMPLETE\',\'n_contexts\':15,\'n_scope_labels\':30,\'n_unique_score_rows\':len(df),\'Table1_rows\':len(tables[\'Table1_complete_benchmark\']),\'n_primary_main_figures\':4,\'n_secondary_main_figures\':4,\'n_primary_full_null_supplements\':3,\'n_failed_candidates\':len(failed),\'new_raw_B_access\':False,\'new_fits_or_seeds\':False,\'biological_directions\':\'BLOCKED\'}\n rp.write_same(out/\'summary.json\',rp.json_bytes(summary))\n memo=\'任务2.6图表解释与阶段性讨论\\n\\nFigure1：M0/M1/M2完整held-out loss及M1/M2相对M0的效应；负差值代表改善。\\nFigure2：真实M1=0，展示全部20个M4 seed的相对loss。代表性contexts为此前固定四组，非本轮择优；补充图完整保留全部15组。M1和每个M4使用相同冻结候选网格、独立T/V调参。\\nFigure3：M2-M3逐seed效应，同时展示zero-weight edge fraction、total weight、beta*lambda_max(Lg)及conditional EDF。保留trace/weight不等于保留全部smoothing强度或自由度。seed区间是随机化描述，不是biological CI。\\nFigure4：真实biological directions仍blocked，因此仅报告两个共享来源的replicate labels，含零线和全部contexts。\\nTable1：每个context/label/method/metric保存loss、M0相对变化、seed数、参数范围和uncertainty说明；完整逐seed原始表另存。\\n\\n阶段性结果：M2相对M1的总体收益很小；M2优于matched随机权重，但需结合smoothing/EDF区别解释。M1与M4不能稳定区分，本次尚未建立PPI edge identity额外价值；应讨论normalization、指标及数据机制。\\n不使用更多random seeds替代机制检查，不启动更复杂edge-weight network或新算法。Dose/sgRNA关联是描述性结果；多guide组合与边际guide层面应区分，不能当因果解释。所有结果留存后供讨论，再决定任务3。\\n\'\n rp.write_same(reports/\'figure_captions_and_discussion.txt\',memo.encode())\n files=[cfg/\'figure_contract.json\',cfg/\'source_effective_parameter_grid.json\',reports/\'figure_captions_and_discussion.txt\']+sorted(raw.glob(\'*.csv\'))+sorted(out.glob(\'*.csv\'))+[out/\'summary.json\']+sorted(figs.glob(\'*\'))\n if self_tests is not None:files.append(cfg/\'self_tests.json\')\n mf={\'files\':[{\'name\':str(p.relative_to(root)),\'sha256\':rp.sha(p)} for p in files]};rp.write_same(out/\'review_manifest.json\',rp.json_bytes(mf));files.append(out/\'review_manifest.json\')\n bundle=root/\'results/summary\'/f\'{name}_review_bundle.zip\'\n with zipfile.ZipFile(bundle,\'w\',zipfile.ZIP_DEFLATED) as z:\n  for p in files:z.write(p,str(p.relative_to(root)))\n  for module in [sys.modules[__name__],rp,cr]:z.write(Path(module.__file__),\'src/\'+Path(module.__file__).name)\n return tables,summary,bundle\n'}
for name,source in SOURCES.items():
    p=ROOT/"src"/(name+".py");p.parent.mkdir(parents=True,exist_ok=True)
    if p.exists() and p.read_text(encoding="utf-8")!=source:
        old=hashlib.sha256(p.read_bytes()).hexdigest()[:12]
        backup=p.with_name(name+"."+old+".backup.py")
        if not backup.exists():backup.write_bytes(p.read_bytes())
    p.write_text(source,encoding="utf-8")
    spec=importlib.util.spec_from_file_location(name,p)
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module
    spec.loader.exec_module(module)
bf=sys.modules["benchmark_figures"]
print("Figure engine:",bf.ENGINE_SHA)


Figure engine: 868c3e38cc0dbc98002a1f4772e57fd10e195a68ee1ce14968fa6e022a0fdd3d


In [3]:
TEST_SOURCE = "import numpy as np\nimport pandas as pd\nimport benchmark_figures as bf\n\ndef run_tests():\n # A complete 30-scope synthetic table, with zero reference boundary retained.\n rows=[]\n for sid in range(30):\n  for m in ['M0','M1','M2','M3','M4']:\n   for seed in range(20) if m in ['M3','M4'] else [None]:\n    loss={'M0':1.,'M1':.9,'M2':.8,'M3':.85,'M4':.9}[m]\n    rows.append({'scope_id':sid,'cell_line':['A','B','C'][sid//10],'drug':str((sid//2)%5),'replicate':f'replicate_{sid%2+1}','n_cells':2,'n_pairs':5000,'method':m,'seed_index':seed,'status':'OK','mean_poisson_deviance':loss,'normalized_mse':loss,'gene_sha256':'g','Lc_sha256':'lc','common_I_row_sha256':'row','alpha':1.,'beta':0. if m=='M0' else 1.,'lambda_rule':'q0.5' if m in ['M2','M3'] else 'fixed_limit','trace_Lg':10.,'weight_sum':5.,'zero_weight_edges':2,'conditional_trace_H':80.,'beta_times_gene_lambda_max':.5,'zero_weight_edge_fraction':.2})\n df=pd.DataFrame(rows);before=df.copy(deep=True);t=bf.calculate_tables(df);table=t['Table1_complete_benchmark']\n assert len(table)==300 and len(t['Figure2_all_null_raw'])==1200 and len(t['Figure3_paired_seed_effects'])==1200\n assert table[table.method.eq('M0')].n_random_seeds.eq(0).all() and table[table.method.eq('M3')].n_random_seeds.eq(20).all()\n assert table[table.method.eq('M0')].seed_q05.isna().all()\n assert np.allclose(table[table.method.eq('M2')].delta_vs_M0_percent,-20.)\n assert np.allclose(t['Figure3_paired_seed_effects'].percent_M2_minus_M3,-100/17)\n assert np.allclose(t['Figure2_all_null_raw'].plot_percent_M4_minus_M1,0.)\n pd.testing.assert_frame_equal(df,before)\n df[bf.METRICS]=0.;z=bf.calculate_tables(df);assert z['Table1_complete_benchmark'].delta_vs_M0_percent.isna().all()\n return {'status':'PASS','checks':['Complete Table1 per scope/method/metric; deterministic vs random uncertainty distinguished','Figure1 M2-vs-M0 baseline; Figure3 paired denominator; Figure2 centering orientation','All seed rows retained; zero reference remains NA; input unchanged']}\nif __name__=='__main__':\n import json\n print(json.dumps(run_tests(),indent=2))\n"
ns={"__name__":"benchmark_figure_tests"}
exec(compile(TEST_SOURCE,"embedded_figure_tests.py","exec"),ns)
TEST_REPORT=ns["run_tests"]()
assert TEST_REPORT["status"]=="PASS"
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "status": "PASS",
  "checks": [
    "Complete Table1 per scope/method/metric; deterministic vs random uncertainty distinguished",
    "Figure1 M2-vs-M0 baseline; Figure3 paired denominator; Figure2 centering orientation",
    "All seed rows retained; zero reference remains NA; input unchanged"
  ]
}


In [4]:
TABLES,SUMMARY,REVIEW_ZIP=bf.run_figures(
    ROOT,FIT_ZIP,REPORT_ZIP,CONTEXT_ZIP,
    name=ANALYSIS_NAME,self_tests=TEST_REPORT)
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))
FIG_DIR=ROOT/"figures"/ANALYSIS_NAME
RAW_DIR=ROOT/"results/raw"/ANALYSIS_NAME


{
  "status": "COMPLETE",
  "n_contexts": 15,
  "n_scope_labels": 30,
  "n_unique_score_rows": 1290,
  "Table1_rows": 300,
  "n_primary_main_figures": 4,
  "n_secondary_main_figures": 4,
  "n_primary_full_null_supplements": 3,
  "n_failed_candidates": 8,
  "new_raw_B_access": false,
  "new_fits_or_seeds": false,
  "biological_directions": "BLOCKED"
}


In [5]:
TABLE1=TABLES["Table1_complete_benchmark"]
PRIMARY="mean_poisson_deviance"
cols=["cell_line","drug","replicate","method","score_mean","delta_vs_M0_absolute","delta_vs_M0_percent","n_random_seeds","seed_q05","seed_q95","beta_min","beta_max"]
print(TABLE1.loc[TABLE1.metric.eq(PRIMARY),cols].to_string(index=False,float_format=lambda v:f"{v:.10g}"))
print("Complete Table 1:",RAW_DIR/"Table1_complete_benchmark.csv")


cell_line       drug   replicate method   score_mean  delta_vs_M0_absolute  delta_vs_M0_percent  n_random_seeds     seed_q05     seed_q95        beta_min        beta_max
     A172    Vehicle replicate_1     M0 0.9160290934                     0                    0               0          NaN          NaN               0               0
     A172    Vehicle replicate_1     M1 0.9160291238       3.040669216e-08       3.31940245e-06               0          NaN          NaN 6.810309248e-06 6.810309248e-06
     A172    Vehicle replicate_1     M2 0.9160200311      -9.062234457e-06     -0.0009892954844               0          NaN          NaN     4.638031206     4.638031206
     A172    Vehicle replicate_1     M3 0.9188230918         0.00279399846         0.3050119783              20  0.918324999 0.9193767506     4.638031206     4.638031206
     A172    Vehicle replicate_1     M4 0.9160291338       4.042668766e-08      4.413253679e-06              20 0.9160291304 0.9160291369 6.810309248e

In [6]:
for p in sorted(FIG_DIR.glob("*.png")):print(p)
print("Complete parameter adjustment rules：",ROOT/"configs"/ANALYSIS_NAME/"source_effective_parameter_grid.json")
print("Complete raw null：",RAW_DIR/"Figure2_all_null_raw.csv")
print("Complete matched pairs：",RAW_DIR/"Figure3_paired_seed_effects.csv")


/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure1_all_context_effects_primary.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure1_all_context_effects_secondary.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure2_PPI_identity_primary.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure2_PPI_identity_secondary.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure2_all_contexts_supplement_01.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure2_all_contexts_supplement_02.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure2_all_contexts_supplement_03.png
/home/featurize/work/biological_prior_validation/figures/task2_four_figures_table1_v1/Figure3_adaptive_attenuation_primary.png
/hom

In [7]:
failure_path=ROOT/"results/summary"/ANALYSIS_NAME/"failed_candidates.csv"
print("Failed Candidate Table：",failure_path)
print("Discussion and Figure Captions：")
print((ROOT/"reports"/ANALYSIS_NAME/"figure_captions_and_discussion.txt").read_text(encoding="utf-8"))


Failed Candidate Table： /home/featurize/work/biological_prior_validation/results/summary/task2_four_figures_table1_v1/failed_candidates.csv
Discussion and Figure Captions：
任务2.6图表解释与阶段性讨论

Figure1：M0/M1/M2完整held-out loss及M1/M2相对M0的效应；负差值代表改善。
Figure2：真实M1=0，展示全部20个M4 seed的相对loss。代表性contexts为此前固定四组，非本轮择优；补充图完整保留全部15组。M1和每个M4使用相同冻结候选网格、独立T/V调参。
Figure3：M2-M3逐seed效应，同时展示zero-weight edge fraction、total weight、beta*lambda_max(Lg)及conditional EDF。保留trace/weight不等于保留全部smoothing强度或自由度。seed区间是随机化描述，不是biological CI。
Figure4：真实biological directions仍blocked，因此仅报告两个共享来源的replicate labels，含零线和全部contexts。
Table1：每个context/label/method/metric保存loss、M0相对变化、seed数、参数范围和uncertainty说明；完整逐seed原始表另存。

阶段性结果：M2相对M1的总体收益很小；M2优于matched随机权重，但需结合smoothing/EDF区别解释。M1与M4不能稳定区分，本次尚未建立PPI edge identity额外价值；应讨论normalization、指标及数据机制。
不使用更多random seeds替代机制检查，不启动更复杂edge-weight network或新算法。Dose/sgRNA关联是描述性结果；多guide组合与边际guide层面应区分，不能当因果解释。所有结果留存后供讨论，再决定任务3。

